In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr


spark = (
    SparkSession.builder.appName("HanduFlow")
    .enableHiveSupport()
    # .config("spark.driver.memory", "4g")
    # .config("spark.executor.memory", "4g")
    # .config("spark.default.parallelism", "4")
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.12:3.1.0,com.databricks:spark-xml_2.12:0.17.0",
    )
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension",
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .getOrCreate()
)

In [8]:
my_df = spark.sql("select * from demo.test")
my_df.orderBy('alpha3_b').show(truncate=False)
my_df.count()


+--------+--------+------+-------------+
|alpha3_b|alpha3_t|alpha2|english      |
+--------+--------+------+-------------+
|USA1    |US1     |US    |Germany      |
|USA10   |US10    |US    |India        |
|USA100  |US100   |US    |United States|
|USA11   |US11    |US    |Canada       |
|USA12   |US12    |US    |United States|
|USA13   |US13    |US    |Germany      |
|USA14   |US14    |US    |India        |
|USA15   |US15    |US    |Canada       |
|USA16   |US16    |US    |United States|
|USA17   |US17    |US    |Germany      |
|USA18   |US18    |US    |India        |
|USA19   |US19    |US    |Canada       |
|USA2    |US2     |US    |India        |
|USA20   |US20    |US    |United States|
|USA21   |US21    |US    |Germany      |
|USA22   |US22    |US    |India        |
|USA23   |US23    |US    |Canada       |
|USA24   |US24    |US    |United States|
|USA25   |US25    |US    |Germany      |
|USA26   |US26    |US    |India        |
+--------+--------+------+-------------+
only showing top

100

In [9]:
my_df_staging = spark.sql("select * from staging.t_full_t_iso_language_codes")
my_df_staging.orderBy('alpha3_b').show(truncate=False)
my_df_staging.count()

+--------+--------+------+-------------+----------------------------------------------------------------+------------------------------------+
|alpha3_b|alpha3_t|alpha2|english      |_x_row_hash                                                     |_x_load_id                          |
+--------+--------+------+-------------+----------------------------------------------------------------+------------------------------------+
|USA1    |US1     |US    |Germany      |1f516ebc96b9bd723b3de2d305254e1297eaff8505d6804a68b8f8693c319845|895f7d75-c2af-48b0-9554-4d7888db6769|
|USA10   |US10    |US    |India        |d5ddac036b0b44e917a8c05f16a68e57b0039190bfdf4042ab07866ec65806e3|895f7d75-c2af-48b0-9554-4d7888db6769|
|USA100  |US100   |US    |United States|5039f0a2b6e462c5df31627885d5b4062620ee638e63bceb6ad5a12684803028|895f7d75-c2af-48b0-9554-4d7888db6769|
|USA11   |US11    |US    |Canada       |23718c1789a7a87edb091112011ca96add8baa09194f60b8f2b643874792222b|895f7d75-c2af-48b0-9554-4d7888db6769|

100

In [10]:
my_df_op = spark.sql("select * from silver.t_iso_language_codes")
my_df_op.orderBy('alpha3_b').show(truncate=False)
my_df_op.count()

+--------+--------+------+-------------+
|alpha3_b|alpha3_t|alpha2|english      |
+--------+--------+------+-------------+
|USA1    |US1     |US    |Germany      |
|USA10   |US10    |US    |India        |
|USA100  |US100   |US    |United States|
|USA11   |US11    |US    |Canada       |
|USA12   |US12    |US    |United States|
|USA13   |US13    |US    |Germany      |
|USA14   |US14    |US    |India        |
|USA15   |US15    |US    |Canada       |
|USA16   |US16    |US    |United States|
|USA17   |US17    |US    |Germany      |
|USA18   |US18    |US    |India        |
|USA19   |US19    |US    |Canada       |
|USA2    |US2     |US    |India        |
|USA20   |US20    |US    |United States|
|USA21   |US21    |US    |Germany      |
|USA22   |US22    |US    |India        |
|USA23   |US23    |US    |Canada       |
|USA24   |US24    |US    |United States|
|USA25   |US25    |US    |Germany      |
|USA26   |US26    |US    |India        |
+--------+--------+------+-------------+
only showing top

100

In [11]:
my_df_op = spark.sql("""
select * from silver.t_iso_language_codes
except
select * from demo.test
""")
my_df_op.show(truncate=False)

+--------+--------+------+-------+
|alpha3_b|alpha3_t|alpha2|english|
+--------+--------+------+-------+
+--------+--------+------+-------+



In [12]:
my_df_op = spark.sql("""
select * from demo.test
except
select * from silver.t_iso_language_codes
""")
my_df_op.show(truncate=False)

+--------+--------+------+-------+
|alpha3_b|alpha3_t|alpha2|english|
+--------+--------+------+-------+
+--------+--------+------+-------+



26/06/13 18:43:32 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
